In [50]:
from ultralytics import YOLO
from trackers import ByteTrackTracker as Tracker
import supervision as sv
import cv2
import numpy as np
class Detector:
    def __init__(self, model: str, framerate=30, track_life = 30, **kwargs):
        self.model = YOLO(model, task='detect')
        self.tracker = Tracker(track_life, framerate, **kwargs)
        self.track_dict = {}

    def __call__(self, frame, **kwargs):
        results = sv.Detections.from_ultralytics(self.model.predict(frame, verbose=False, **kwargs)[0])
        dets = self.tracker.update(results)
        return dets

    def only_track(self, frame, ids=(), **kwargs):
        results = sv.Detections.from_ultralytics(self.model.predict(frame, verbose=False, **kwargs)[0])
        # TODO: add protections for empty results
        tracked_dets = self.tracker.update(results[np.isin(results.class_id, ids)])
        untracked_dets = results[np.isin(results.class_id, ids, invert=True)]
        return tracked_dets, untracked_dets

    def separate(self, dets=None, frame=None, **kwargs):
        if dets is None:
            if frame is None:
                raise ValueError("Either dets or frame must be provided.")
            dets = self(frame, **kwargs)
        ppe_dets = dets[np.isin(dets.class_id, (0, 1, 2, 3, 4, 7))]
        cone_dets = dets[dets.class_id == 6]

        person_dets = dets[dets.class_id == 5]
        machinery_dets = dets[dets.class_id == 8]
        vehicle_dets = dets[dets.class_id == 9]

        return {"ppe": ppe_dets, "person": person_dets, "cone": cone_dets,
                "machinery": machinery_dets, "vehicle": vehicle_dets}

    def feature_from_moving(self, det_list: list[sv.Detections]):  # TODO: pass ppe dets here and match with person dets
        nodes = []
        current_centroids = {}
        for detections in det_list:
            subset = []
            if len(detections) > 0:
                # Vectorized centroid calculation for performance
                cx = (detections.xyxy[:, 0] + detections.xyxy[:, 2]) / 2.0
                cy = (detections.xyxy[:, 1] + detections.xyxy[:, 3]) / 2.0

                for i in range(len(detections)):
                    tracker_id = detections.tracker_id[i]
                    class_id = detections.class_id[i]

                    # Skip detections without an assigned track ID
                    if tracker_id is None or tracker_id == -1:
                        continue

                    icx, icy = cx[i], cy[i]
                    current_centroids[tracker_id] = (icx, icy)

                    # Calculate velocity delta [vx, vy]
                    vx, vy = 0.0, 0.0
                    if tracker_id in self.track_dict:
                        px, py = self.track_dict[tracker_id]
                        vx = icx - px
                        vy = icy - py
                    subset.append([int(tracker_id), int(class_id), icx, icy, vx, vy])
            nodes.append(np.array(subset))

        # Roll over state for the next frame
        self.track_dict = current_centroids
        return nodes

    def cone_features(self, det_list: list[sv.Detections]):
        nodes = []
        for detections in det_list:
            if np.all(detections.class_id != 6):
                continue
            if len(detections) == 0:
                continue
            cx = (detections.xyxy[:, 0] + detections.xyxy[:, 2]) / 2.0
            cy = (detections.xyxy[:, 1] + detections.xyxy[:, 3]) / 2.0
            vecs = np.pad(np.stack((cx, cy), axis=1), ((0, 0), (0, 2)), "constant", constant_values=0)
            vecs[:, 0] = -1
            vecs[:, 1] = 6
            return vecs

    def match_ppe(self, person_dets, ppe_dets):
        pass

    def get_vectors(self):
        pass

In [51]:
det = Detector("../assets/ppe_50ep.engine")
v = cv2.VideoCapture("../assets/people-walking.mp4")

for i in range(5):
    ret, frame = v.read()
    if not ret:
        break
    td, ud = det.only_track(frame, ids=(5, 8, 9))
    moving = det.separate(td)
    static = det.separate(ud)
    # in a proper dataset video, it would show people, machinery and vehicles tracked.
    # len(moving) is the length of full detections (5), but only filled on tracked values (3)
    moving_dets = det.feature_from_moving(list(moving.values()))
    # ditto for static, but only filled in cone and ppe detections (if we do implement it.
    static_dets = det.feature_from_static(list(static.values()))  # really only needs to return cones.

    print(moving_dets, end="\n\n")

Loading ../assets/ppe_50ep.engine for TensorRT inference...
[09/07/2026-01:12:46] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[09/07/2026-01:12:46] [TRT] [I] Loaded engine size: 160 MiB
[09/07/2026-01:12:46] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +58, now: CPU 2, GPU 645 (MiB)
[array([], dtype=float64), array([], dtype=float64), array([], dtype=float64), array([], dtype=float64), array([], dtype=float64)]

[array([], dtype=float64), array([[          0,           5,           0,           0],
       [          1,           5,           0,           0],
       [          2,           5,           0,           0],
       [          3,           5,           0,           0],
       [          4,           5

In [59]:
moving

{'ppe': Detections(xyxy=array([], shape=(0, 4), dtype=float32), mask=None, confidence=array([], dtype=float32), class_id=array([], dtype=int64), tracker_id=array([], dtype=int64), data={'class_name': array([], dtype='<U14')}, metadata={}),
 'person': Detections(xyxy=array([[     673.88,         618,      751.88,      807.75],
        [     758.25,      428.25,         828,      590.25],
        [     1409.2,         696,      1520.2,       886.5],
        [       1689,       66.75,        1794,         300],
        [     1465.5,      479.25,        1554,      647.25],
        [     1380.8,           0,        1455,       184.5],
        [     1453.5,      321.75,      1529.2,       481.5],
        [     711.75,      221.62,       769.5,       367.5],
        [        249,       442.5,      327.56,      612.75],
        [       1143,      915.75,        1251,      1077.8],
        [       1821,           0,        1857,       50.25],
        [      562.5,      201.38,      638.62,     